# Week 7 — Retries & Error Handling

## Learning Goals

- Understand why AI workflows fail
- Understand retry policies
- Configure retries for LangGraph nodes
- Understand retry limits
- Build a fallback workflow

In [1]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy

In [2]:
class State(TypedDict):
    question: str
    answer: int

In [15]:
attempt_count = 0

def unrelieable_node(state: State):
    global attempt_count
    attempt_count += 1
    print(f"Attempt {attempt_count}: Unreliable node is running...")
    
    if attempt_count < 3:
        raise Exception("Simulated failure in unreliable node.")
    
    return {
        "answer": "Life is a gift, and the answer to life, the universe, and everything is PRESENT."
    }

In [18]:
try:
    result = unrelieable_node({"question": "What is the answer to life, the universe, and everything?"})
    print("Result:", result)
except Exception as e:
    print("Error:", e)

Attempt 3: Unreliable node is running...
Result: {'answer': 'Life is a gift, and the answer to life, the universe, and everything is PRESENT.'}


In [20]:
graph = StateGraph(State)

graph.add_node("unreleiable_node", unrelieable_node, retry_policy=RetryPolicy(max_attempts=3))

graph.add_edge(START,"unreleiable_node")
graph.add_edge("unreleiable_node", END)

app = graph.compile()

In [24]:
attempt_count = 0  # Reset attempt count for the graph execution

result = app.invoke({
    "question": "What is the answer to life, the universe, and everything?",
    "result": ""
})

result

Attempt 1: Unreliable node is running...
Attempt 2: Unreliable node is running...
Attempt 3: Unreliable node is running...


{'question': 'What is the answer to life, the universe, and everything?',
 'answer': 'Life is a gift, and the answer to life, the universe, and everything is PRESENT.'}

In [28]:
class State(TypedDict):
    question: str
    retrieval_status: int
    result: str

In [ ]:
def retrieval_node(state: State):
    print("Retrieval node running...")

    return {
        "retrieval_status": "failed",
        "result": ""
    }

In [30]:
def fallback_node(state: State):
    print("Fallback node running...")

    return {
        "result": "Using fallback information."
    }

In [31]:
def success_node(state: State):
    print("Success node running...")

    return {
        "result": "Retrieval completed successfully."
    }

In [32]:
def route_retrieval(state: State):
    if state["retrieval_status"] == "failed":
        return "fallback"

    return "success"

In [38]:
graph = StateGraph(State)
graph.add_node("retrieval", retrieval_node)
graph.add_node("fallback", fallback_node)
graph.add_node("success", success_node)

graph.add_edge(START, "retrieval")
graph.add_conditional_edges("retrieval", route_retrieval)
graph.add_edge("fallback", END)
graph.add_edge("success", END)

app = graph.compile()

In [39]:
result = app.invoke({
    "question": "What is the answer to life, the universe, and everything?",
    "retrieval_status": "",
    "result": ""
})

result

Retrieval node running...
Fallback node running...


{'question': 'What is the answer to life, the universe, and everything?',
 'retrieval_status': 'failed',
 'result': 'Using fallback information.'}